# Scrapping the ASI Conference Website
**Note:** For reproducibility:



1.   First make a copy of the notebook and work on the copy
2.   Check these things: paths name, file names, APIs IDs, and customize them according to your environment
3.   After doing a major data operation, you can push your final version here: https://github.com/GaelKBertrand/scholar_info_append






### Mounting Google Drive as a working director

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Installing and loading all dependencies and libraries needed

In [ ]:
#Install necessary dependencies and libraries
!pip install waybackpy
!pip install googlesearch-python
!pip install google-api-python-client
!pip install beautifulsoup4
!pip install fake_useragent #necessary for google search rapid automation
!pip install selenium
!apt-get update
!apt install -y chromium-chromedriver
!apt-get install -y wget
!pip install requests-html

import requests
from bs4 import BeautifulSoup
import pandas as pd
from time import sleep
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import logging


from waybackpy import WaybackMachineSaveAPI
from googlesearch import search
from googleapiclient.discovery import build
import time
import requests
from fake_useragent import UserAgent
from bs4 import BeautifulSoup
import logging

#from requests_html import HTMLSession
#from requests_html import AsyncHTMLSession
import pandas as pd
import logging
import asyncio
import nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.0/201.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.7/481.7 kB 28.8 MB/s eta 0:00:00
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,190 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,523 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:10 https://ppa.launchpadcontent.ne

# Scrapping ISI: 3rd attempt: PDF MINER.SIX

## Extracting text

In [2]:
!pip install pdfminer.six
from pdfminer.high_level import extract_text

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 35.9 MB/s eta 0:00:00


In [12]:
# Specify the path to your PDF in Google Drive
pdf_path = '/content/drive/My Drive/2023.pdf'

In [13]:
text = extract_text(pdf_path)
print(text)


Parallel Sessions 1                                                18 December, Monday 11:50 AM- 01:00  PM    
Session 1(a) Social Identity and Financial Inclusion (Conference Room) 
Chair: Sandhya Garg 
Rikhia Bhukta (Indian Institute of Technology Kanpur), Debayan Pakrashi (Indian Statistical Institute Kolkata; 
Indian Institute of Technology Kanpur), Sarani Saha (Indian Institute of Technology Kanpur), and Ashish Sedai 
(University of Texas at Arlington): "Does Financial Inclusion Mitigate Social Exclusion? Causal Evidence from 
India" 

Sandhya Garg (Institute of Economic Growth), Samarth Gupta (Ahmedabad University), and Sushanta Mallick 
(Queen Mary University of London): "Does Social Identity Constrain Rural Entrepreneurship? The Role of 
Financial Inclusion" 
Session 1(b) Welfare Programmes I (Seminar Room 1) 
Vidhya Unnikrishnan (University of Manchester): "Impact of Social Protection Policy Targeted on the 
Disabled: Panel Data Evidence from India" 

Chair: Michael Palmer 

M

Remove 'Chair' sections

In [14]:
import re
# Clean the text by removing lines that start with 'Chair:'
text = re.sub(r"Chair:.*\n", "", text)

# Print the cleaned text
print(text)


Parallel Sessions 1                                                18 December, Monday 11:50 AM- 01:00  PM    
Session 1(a) Social Identity and Financial Inclusion (Conference Room) 
Rikhia Bhukta (Indian Institute of Technology Kanpur), Debayan Pakrashi (Indian Statistical Institute Kolkata; 
Indian Institute of Technology Kanpur), Sarani Saha (Indian Institute of Technology Kanpur), and Ashish Sedai 
(University of Texas at Arlington): "Does Financial Inclusion Mitigate Social Exclusion? Causal Evidence from 
India" 

Sandhya Garg (Institute of Economic Growth), Samarth Gupta (Ahmedabad University), and Sushanta Mallick 
(Queen Mary University of London): "Does Social Identity Constrain Rural Entrepreneurship? The Role of 
Financial Inclusion" 
Session 1(b) Welfare Programmes I (Seminar Room 1) 
Vidhya Unnikrishnan (University of Manchester): "Impact of Social Protection Policy Targeted on the 
Disabled: Panel Data Evidence from India" 


Michael Palmer (University of Western Austral

Insert Paper Markers

In [15]:
import re

# Define the function to add 'begin paper' and 'end paper' markers
def add_paper_markers(text):
    # Add 'paper' after each paper title
    text = re.sub(r'("[^"]+")', r'\1 paper', text)

    # Add 'begin paper' before the first author
    text = re.sub(r'(\n[A-Za-z\s]+ \([^)]+\))', r'\nbegin paper \1', text)

    # Replace ' paper' with ' end paper' after the title
    text = re.sub(r'(".*?") paper', r'\1 end paper', text)

    return text

# Apply the transformation to the input text
text_with_markers = add_paper_markers(text)

# Print the transformed text with markers
print(text_with_markers)


Parallel Sessions 1                                                18 December, Monday 11:50 AM- 01:00  PM    
Session 1(a) Social Identity and Financial Inclusion (Conference Room) 
begin paper 
Rikhia Bhukta (Indian Institute of Technology Kanpur), Debayan Pakrashi (Indian Statistical Institute Kolkata; 
Indian Institute of Technology Kanpur), Sarani Saha (Indian Institute of Technology Kanpur), and Ashish Sedai 
(University of Texas at Arlington): "Does Financial Inclusion Mitigate Social Exclusion? Causal Evidence from 
India" paper 
begin paper 

Sandhya Garg (Institute of Economic Growth), Samarth Gupta (Ahmedabad University), and Sushanta Mallick 
(Queen Mary University of London): "Does Social Identity Constrain Rural Entrepreneurship? The Role of 
Financial Inclusion" paper 
Session 1(b) Welfare Programmes I (Seminar Room 1) 
begin paper 
Vidhya Unnikrishnan (University of Manchester): "Impact of Social Protection Policy Targeted on the 
Disabled: Panel Data Evidence from Indi

In [7]:
!pip install pandas openpyxl spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 51.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [16]:
import pandas as pd
import re
import spacy

# Load SpaCy NLP model
nlp = spacy.load("en_core_web_sm")

# File paths
excel_path = "/content/drive/My Drive/ISI/ISI2023.xlsx"
output_excel_path = "/content/drive/My Drive/ISI/ISI2023.xlsx"

# Load the Excel file
excel_data = pd.ExcelFile(excel_path)

# Select the specific sheet
sheet_name = "Chair of regular sessions"
df = excel_data.parse(sheet_name)

# Ensure 'Author' and 'Institute/Affiliation' columns exist
if "Name of Chair" in df.columns:
    if "Institute/Affiliation" not in df.columns:
        df["Institute/Affiliation"] = ""

# Process the text
pdf_text = text
doc = nlp(pdf_text)

# Iterate through each row in the DataFrame
for index, row in df.iterrows():
    author_name = row["Name of Chair"]
    pattern = re.compile(rf"\b{author_name}\b \((.*?)\)")
    match = pattern.search(pdf_text)
    if match:
        affiliation = match.group(1)
        df.at[index, "Institute/Affiliation"] = affiliation

# Save the updated DataFrame back to the Excel file
with pd.ExcelWriter(excel_path, mode="a", engine="openpyxl", if_sheet_exists="overlay") as writer:
    df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Updated Excel file saved as {output_excel_path}")


Updated Excel file saved as /content/drive/My Drive/ISI/ISI2023.xlsx
